In [16]:
import pandas as pd
import numpy as np
import warnings # control warning messages because these libraries give lots of warnings which are not actually the error and only end up cluttering our notebook.

# now to ignore the warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/application_train_clean.csv")
print(df.shape)
df.head()

(307511, 123)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_ANOM
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,False
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,False
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False


In [17]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH']) / 365

print(df['AGE_YEARS'].describe())

count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: AGE_YEARS, dtype: float64


In [18]:
df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED']) / 365

print(df['EMPLOYMENT_YEARS'].describe())

count    252137.000000
mean          6.531971
std           6.406466
min          -0.000000
25%           2.101370
50%           4.515068
75%           8.698630
max          49.073973
Name: EMPLOYMENT_YEARS, dtype: float64


In [19]:
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

print(df['CREDIT_INCOME_RATIO'].describe())

count    307511.000000
mean          3.963758
std           2.686391
min           0.095238
25%           2.027183
50%           3.275862
75%           5.165557
max          84.736842
Name: CREDIT_INCOME_RATIO, dtype: float64


In [20]:
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

print(df['ANNUITY_INCOME_RATIO'].describe())

count    307499.000000
mean          0.181223
std           0.094399
min           0.008700
25%           0.115267
50%           0.162933
75%           0.229156
max           1.875965
Name: ANNUITY_INCOME_RATIO, dtype: float64


In [21]:
df['HAS_HOUSING_DATA'] = df['APARTMENTS_AVG'].notnull().astype(int)

print(df['HAS_HOUSING_DATA'].value_counts())

HAS_HOUSING_DATA
0    156061
1    151450
Name: count, dtype: int64


In [22]:
df['HAS_BUREAU_INQUIRY_DATA'] = df['AMT_REQ_CREDIT_BUREAU_YEAR'].notnull().astype(int)

print(df['HAS_BUREAU_INQUIRY_DATA'].value_counts())

HAS_BUREAU_INQUIRY_DATA
1    265992
0     41519
Name: count, dtype: int64


In [23]:
df['EXT_SOURCE_1_WAS_MISSING'] = df['EXT_SOURCE_1'].isnull().astype(int)
df['EXT_SOURCE_3_WAS_MISSING'] = df['EXT_SOURCE_3'].isnull().astype(int)

print(df['EXT_SOURCE_1_WAS_MISSING'].value_counts())
print(df['EXT_SOURCE_3_WAS_MISSING'].value_counts())

EXT_SOURCE_1_WAS_MISSING
1    173378
0    134133
Name: count, dtype: int64
EXT_SOURCE_3_WAS_MISSING
0    246546
1     60965
Name: count, dtype: int64


In [24]:
print(f"Original shape: (307511, 123)")
print(f"New shape: {df.shape}")
print(f"New columns added: {df.shape[1] - 123}")

Original shape: (307511, 123)
New shape: (307511, 131)
New columns added: 8


In [25]:
df.to_csv("../data/application_train_features.csv", index=False)
print("Exported successfully")
print(df.shape)

Exported successfully
(307511, 131)


# NOW Let's see basline model with feature engneering.


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X = df.drop(columns="TARGET")
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

Training set: (246008, 130)
Testing set : (61503, 130)


In [27]:
numerical_features = X_train.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape :", X_test_processed.shape)

Processed Training Shape: (246008, 254)
Processed Testing Shape : (61503, 254)


In [28]:
log_reg_features = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
log_reg_features.fit(X_train_processed, y_train)

y_pred_features = log_reg_features.predict(X_test_processed)
y_prob_features = log_reg_features.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_features))
print("Precision:", precision_score(y_test, y_pred_features))
print("Recall   :", recall_score(y_test, y_pred_features))
print("F1 Score :", f1_score(y_test, y_pred_features))
print("ROC AUC  :", roc_auc_score(y_test, y_prob_features))

Accuracy : 0.687738809488968
Precision: 0.16117826211097364
Recall   : 0.6821752265861027
F1 Score : 0.26074906655375496
ROC AUC  : 0.7504050765062181
